<div style="border-left:6px solid #00897B;padding:12px 18px;background:#F5F5F5">
<span style="color:#00897B;font-weight:bold">PAUL MINCHELLA · ONLINE LEARNING · STUDENT TEMPLATE</span>
<h1 style="color:#283593">Learning one day at a time</h1>
<b>OGD, OGD + Ridge and the newsvendor loss on a real demand stream</b><br>
Questions, functions to complete and written answers · Suggested duration: 2h30
</div>

This practical accompanies [the course slides](../Cours_Online_Learning.pdf). We forecast the daily number of
bikes rented in Washington, D.C. (Capital Bikeshare) and we learn **after**, never before, each observation is revealed.
The dataset is small enough to write every algorithm by hand, without hiding the updates in a library.

<div style="border-left:4px solid #00897B;padding:8px 14px;background:#E0F2F1">
<b style="color:#283593">How to use this notebook</b><br>

- Complete the bodies of the functions and classes marked <code># To complete</code>. Their <b>names, signatures and docstrings are mandatory</b>:
  the rest of the notebook calls them.
- Workspace cells contain comments saying what to compute and in which variables to store the result.
  Keep these names (<code>X_train</code>, <code>results</code>, <code>scorecard</code>…), since later cells reuse them.
- Write your answers in the <b>Your answer:</b> cells, in a few sentences.
- <b>Never</b> choose a hyperparameter by looking at the test period (2012).
- Before handing in the notebook: <i>Kernel → Restart & Run All</i>, and check that it runs without errors.
</div>

### Learning outcomes

By the end of this practical, you should be able to:

1. describe a time series and identify the variables available **before** the decision;
2. check the assumptions behind regret guarantees (compact domain, bounded covariates and target);
3. implement projected OGD for the squared loss, with and without ridge;
4. implement OGD for the newsvendor loss and observe that the optimum is the critical quantile $\tau = c_b/(c_b+c_h)$;
5. evaluate properly (train then test, predict before learning) with RMSE and average regret.

### Plan

| Part | Content | Slides |
|:--|:--|:--|
| **0** | Packages and data | — |
| **1** | Descriptive statistics | 9–14 |
| **2** | Online learning: assumptions, covariates, protocol, OGD, Ridge, Newsvendor | protocol 15–25; regret 26–29; OGD 44–49; regression 53–59; inventory 60–64 |
| **3** | Final metrics: RMSE, average regret, conclusion | 26–29, 65 |

<div style="border-left:4px solid #42A5F5;padding:8px 14px;background:#E3F2FD">
<b style="color:#283593">Reminder — moving average</b><br>
A moving average of window $w$ replaces each value by the mean of $w$ neighbouring values. It smooths day-to-day noise
to reveal the trend and seasonality.

- <b>Trailing</b> (causal) moving average, which uses only the past:
$\displaystyle \bar y_t^{\,\text{trail}} = \frac1w \sum_{k=0}^{w-1} y_{t-k}.$
- <b>Centred</b> moving average, which uses as many days after $t$ as before:
$\displaystyle \bar y_t^{\,\text{centre}} = \frac1w \sum_{k=-\lfloor w/2 \rfloor}^{\lceil w/2 \rceil - 1} y_{t+k}.$

The centred version is better aligned with the series, so it is the right tool to <i>describe</i> it. It looks into the future,
so it must <b>never</b> be used to <i>forecast</i>. In pandas: <code>s.rolling(w).mean()</code> (trailing) and
<code>s.rolling(w, center=True).mean()</code> (centred).
</div>

### Name and date

...

# Part 0 — Packages and data

We import only what we need: NumPy for linear algebra, pandas for the table, Matplotlib for figures, and two SciPy
functions (a root finder and a linear-programming solver) that are used only to compute the **hindsight comparators**
of Part 3. The online algorithms are written by hand.

*These two cells are provided: simply run them.*

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq, linprog
from IPython.display import display

DATA_DIR = Path.cwd() / "online_learning_bike_sharing_data"
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

PALETTE = {
    "teal": "#00897B",
    "skyblue": "#81D4FA",
    "blue": "#42A5F5",
    "darkblue": "#283593",
    "orange": "#FB8C00",
    "purple": "#8E24AA",
}
# One colour and one line style per model (the line style also helps black-and-white printing).
COLORS = {
    "Naive (yesterday)": PALETTE["skyblue"],
    "Naive (D-7)": PALETTE["teal"],
    "OGD": PALETTE["blue"],
    "OGD + Ridge": PALETTE["darkblue"],
    "Newsvendor (no covariates)": PALETTE["orange"],
    "Newsvendor (contextual)": PALETTE["purple"],
}
STYLES = {
    "Naive (yesterday)": ":", "Naive (D-7)": ":",
    "OGD": "-", "OGD + Ridge": "-.",
    "Newsvendor (no covariates)": "--", "Newsvendor (contextual)": "-",
}
plt.rcParams.update({
    "figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "axes.titleweight": "bold", "axes.labelcolor": "#333333", "lines.linewidth": 2.0,
    "axes.prop_cycle": plt.cycler(color=[PALETTE["teal"], PALETTE["blue"], PALETTE["orange"]]),
})
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

In [ ]:
def load_daily_data(data_dir=DATA_DIR, url=DATA_URL):
    """Load day.csv: local cache if present, otherwise download the UCI archive."""
    data_dir.mkdir(parents=True, exist_ok=True)
    csv_path = data_dir / "day.csv"
    if not csv_path.exists():
        print(f"Downloading from {url} ...")
        with urlopen(url, timeout=60) as response:
            archive = ZipFile(BytesIO(response.read()))
        csv_path.write_bytes(archive.read("day.csv"))  # read only the expected member
    return pd.read_csv(csv_path, parse_dates=["dteday"])


daily = load_daily_data()
print("Data loaded from:", DATA_DIR / "day.csv")

# Part 1 — Descriptive statistics

**Dataset:** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset),
DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894), licence CC BY 4.0. We use `day.csv`:
one row per day, from 1 January 2011 to 31 December 2012. Requested citation: Hadi Fanaee-T and João Gama (2013),
*Event labeling combining ensemble detectors and background knowledge*, Progress in Artificial Intelligence.

The variable to forecast is `cnt`, the total number of bikes rented during the day.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.1 — Load and display</b><br>
Display the first rows of the dataset and its shape <code>(n_row, n_col)</code>.
</div>

In [ ]:
# To complete: display the first rows of `daily` and its shape (n_row, n_col).

**Your answer:** ...

*How many rows and columns? Into which groups can the columns be sorted?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.2 — Statistical summary</b><br>
For each numerical variable, compute the mean, standard deviation, minimum, median, maximum, number of missing values and number of observations. Also check that dates are unique and consecutive, and that <code>cnt = casual + registered</code>.
</div>

In [ ]:
def summary_table(frame):
    """
    Statistical summary of the numerical variables.

    Parameters
    ----------
    frame : pandas.DataFrame
        Raw daily data (output of load_daily_data).

    Returns
    -------
    table : pandas.DataFrame
        One row per numerical variable (without the `instant` column) and the columns
        "mean", "std", "min", "median", "max", "n_nan", "n_obs", rounded to 3 decimals.
    """
    # To complete
    ...


# To complete:
# 1) display summary_table(daily);
# 2) display the period covered (first and last date);
# 3) check that dates are unique and consecutive (a gap of exactly one day);
# 4) check that cnt = casual + registered on every day.

**Your answer:** ...

*Are there missing values? What is the order of magnitude of `cnt`? What does the identity `cnt = casual + registered` imply for the choice of covariates? How should the means of `holiday` and `workingday` be read?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.3 — Plot the series</b><br>
Plot the observed series <code>cnt</code> against the date. What do you observe?
</div>

In [ ]:
# To complete: plot `cnt` against `dteday`, with
#   - the daily series (thin line, PALETTE["skyblue"]);
#   - a 28-day centred moving average (PALETTE["darkblue"]), see the reminder in the preamble;
#   - a vertical line on 1 January 2012 (PALETTE["orange"], dashed).
# Title, axis labels and legend are required. Also print the date and value of the minimum of `cnt`.

**Your answer:** ...

*Trend? Seasonality? Variability? Extreme values? Why must the centred moving average not be used to forecast?*

### Covariates of `day.csv` at a glance

| Column | Description |
|:--|:--|
| `instant` | Row index, from 1 to 731. |
| `dteday` | Calendar date of the observation. |
| `season` | Season: 1 = winter, 2 = spring, 3 = summer, 4 = autumn. |
| `yr` | Year: 0 = 2011, 1 = 2012. |
| `mnth` | Month, from 1 to 12. |
| `holiday` | 1 if the day is a public holiday, 0 otherwise. |
| `weekday` | Day of the week, from 0 (Sunday) to 6 (Saturday). |
| `workingday` | 1 if the day is neither a weekend nor a holiday, 0 otherwise. |
| `weathersit` | Weather situation, from 1 (clear) to 4 (heavy rain, snow, storm). |
| `temp` | Temperature in °C, divided by 41 (the maximum). |
| `atemp` | Feeling temperature in °C, divided by 50 (the maximum). |
| `hum` | Relative humidity, divided by 100. |
| `windspeed` | Wind speed, divided by 67 (the maximum). |
| `casual` | Number of rentals by casual (unregistered) users. |
| `registered` | Number of rentals by registered users. |
| `cnt` | Total rentals, `casual` + `registered`: **the target**. |

# Part 2 — Online learning

## 2.1 Protocol and regret

We play a repeated game against nature. On each day $t = 1, 2, \dots, T$:

1. we observe the covariates $\mathbf x_t \in \mathbb R^d$, all known **before** the decision;
2. we choose a parameter $\theta_t \in \mathcal K$ and predict $\widehat y_t = \mathbf x_t^\top \theta_t$ (or order a quantity $q_t$);
3. the demand $y_t$ is revealed and we pay the loss $\ell_t(\theta_t)$;
4. we correct: $\theta_{t+1} = \Pi_{\mathcal K}\big(\theta_t - \eta_t\, g_t\big)$ with $g_t \in \partial \ell_t(\theta_t)$.

This is **projected Online Gradient Descent** (OGD). For the ball $\mathcal K = \{\theta : \|\theta\|_2 \le B\}$, the projection is explicit:
$\Pi_{\mathcal K}(z) = z \cdot \min\{1,\ B/\|z\|_2\}$.

Quality is measured by the **regret**: the gap between our cumulative loss and that of the best **fixed** parameter chosen in hindsight,
$$
R_T \;=\; \sum_{t=1}^T \ell_t(\theta_t) \;-\; \min_{u \in \mathcal K} \sum_{t=1}^T \ell_t(u).
$$
No probabilistic assumption is made on $(y_t)$: the sequence can be arbitrary, even adversarial.
An algorithm is *no-regret* if $R_T/T \to 0$: on average, it asymptotically does as well as the best fixed choice.

## 2.2 Two families of models

**(i) Squared loss: OGD and OGD + Ridge** (slides 53–59).

| | Loss | Gradient | Update | Guarantee |
|:--|:--|:--|:--|:--|
| OGD | $\ell_t(\theta) = \tfrac12(\mathbf x_t^\top\theta - y_t)^2$ | $e_t\,\mathbf x_t$, with $e_t = \widehat y_t - y_t$ | $\Pi_{\mathcal K}(\theta_t - \eta\, e_t \mathbf x_t)$ | $R_T \le D\,G\,\sqrt T$ for $\eta = \frac{D}{G\sqrt T}$ |
| OGD + Ridge | $\widetilde\ell_t(\theta) = \ell_t(\theta) + \tfrac{\lambda}{2}\|\theta\|_2^2$ | $e_t\,\mathbf x_t + \lambda\theta$ | $\Pi_{\mathcal K}\big((1-\eta_t\lambda)\theta_t - \eta_t\, e_t \mathbf x_t\big)$ | $\widetilde R_T \le \frac{G_\lambda^2}{2\lambda}(1+\log T)$ for $\eta_t = \frac{1}{\lambda t}$ |

with $D = 2B$ (the diameter of $\mathcal K$), $G = (RB+Y)R$ and $G_\lambda = G + \lambda B$.

- **OGD:** the residual sets the sign and size of the correction, $\mathbf x_t$ sets its direction. The loss is convex,
  but its Hessian $\mathbf x_t\mathbf x_t^\top$ has rank 1: no strong convexity, hence a $\sqrt T$ regret.
- **Ridge:** the Hessian becomes $\mathbf x_t\mathbf x_t^\top + \lambda I_d \succeq \lambda I_d$. The loss is $\lambda$-strongly convex
  in **every** direction, hence a logarithmic regret. Beware: this guarantee concerns the **regularised** loss
  $\widetilde\ell_t$, not the squared error alone.

**(ii) Newsvendor loss: ordering before the demand is known** (slides 60–64).

We order $q_t \in [0, Q_{\max}]$ before observing $y_t$. Each excess bike costs $c_h$, each missing bike costs $c_b$:
$$
\ell_t(q) = c_h\,(q - y_t)_+ + c_b\,(y_t - q)_+ = (c_b + c_h)\,\rho_\tau(y_t - q),
\qquad \tau = \frac{c_b}{c_b + c_h},
$$
where $\rho_\tau(r) = \tau r_+ + (1-\tau)(-r)_+$ is the *pinball* loss. A subgradient is $g_t = c_h$ in case of excess ($q_t > y_t$),
$g_t = -c_b$ in case of shortage ($q_t < y_t$) and $0$ in case of equality. Hence the rule: **shortage → order more, excess → order less**,
asymmetrically. With $\eta = Q_{\max}/(G\sqrt T)$ and $G = \max(c_b, c_h)$, we get $R_T \le Q_{\max}\, G\, \sqrt T$.

**Key result:** for a demand with distribution $F$, the expected cost is minimised at the **critical quantile** $q^\star = F^{-1}(\tau)$.
On a sample, the best constant order is the empirical $\tau$-quantile. We will check this numerically.

The dataset does **not** provide costs: we **set** $c_b = 3$ and $c_h = 1$ (a missing bike loses a rental, which is
three times as costly as an idle bike), so $\tau = 0.75$, as in the course. We will also test a
**contextual** version $q_t = \mathbf x_t^\top\theta_t$, which learns a conditional quantile with the same subgradient multiplied by $\mathbf x_t$.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.1 — Do the assumptions apply?</b><br>
Regret guarantees rest on assumptions. For each one, ask yourself <b>why</b> it is needed in the proof and <b>how</b> to guarantee it on our data: is the domain $\mathcal K$ convex and compact? Are the covariates bounded, $\|\mathbf x_t\|_2 \le R$? Is the target bounded, $|y_t| \le Y$? Are the losses convex? Which assumptions are <b>not</b> needed?
</div>

**Your answer:** ...

*For each assumption: its role in the proof, and how you guarantee it here. Which assumptions are not needed?*

In [ ]:
# Choices declared a priori (provided, do not modify)
P = 7                      # number of demand lags used as covariates
TARGET_SCALE = 10_000.0    # scale declared a priori: y = cnt / 10,000
B = 2.0                    # radius of K = {θ : ||θ||₂ ≤ B}
C_B, C_H = 3.0, 1.0        # chosen costs: a shortage is 3× as costly as an excess
TAU = C_B / (C_B + C_H)    # critical quantile
Q_MAX = 1.0                # newsvendor capacity: 10,000 bikes in normalised units

# To complete: constants of the theorems, derived from the construction of the covariates (Question 2.2)
R = ...       # bound on ||x_t||₂
Y = ...       # bound on |y_t|
D = ...       # diameter of K
G_SQ = ...    # gradient bound of the squared loss on K
G_NV = ...    # newsvendor subgradient bound, without covariates
G_CTX = ...   # newsvendor subgradient bound, contextual


def theory_table(T):
    """
    Theoretical steps and average-regret bounds for a horizon T.

    Parameters
    ----------
    T : int
        Horizon (number of rounds).

    Returns
    -------
    table : pandas.DataFrame
        Rows "OGD (squared)", "Newsvendor (no covariates)", "Newsvendor (contextual)";
        columns "D", "G", "η* = D/(G√T)" and "bound R_T/T = DG/√T".
    """
    # To complete
    ...


# To complete:
# 1) print R, Y, B and τ, and check a posteriori that max(cnt) / TARGET_SCALE ≤ Y;
# 2) display theory_table(T=365);
# 3) for λ ∈ {0.01, 0.1}, compute the ridge bound G_λ²(1 + log T)/(2λT) with G_λ = G_SQ + λB and T = 365.

**Your answer:** ...

*Comment on the order of magnitude of these bounds compared with squared losses of order 10⁻³. What do they really guarantee?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.2 — Prepare the covariates</b><br>
For each day $t$, build a vector $\mathbf x_t = (1, \mathbf z_t, y_{t-1}, \dots, y_{t-p})$ that contains only information available <b>before</b> $y_t$ is observed. Here $\mathbf z_t$ gathers the calendar variables known in advance. The learned parameter is $\theta_t$, and we predict $\widehat y_t = \mathbf x_t^\top\theta_t$. Which columns must be excluded, and why? Check numerically that $\|\mathbf x_t\|_2 \le R$ and that the covariates are causal.
</div>

**Required specification.** $\mathbf x_t$ contains, in this order: `intercept` (= 1), `workingday`, `holiday`,
`sin_annual` $=\sin(2\pi j_t/365.25)$, `cos_annual` $=\cos(2\pi j_t/365.25)$ where $j_t$ is the day of the year,
then the lags `y_lag1`, …, `y_lag7` of $y = \texttt{cnt}/10{,}000$. Days without a complete history are dropped.

**Your answer:** ...

*Why these covariates? Which columns of `day.csv` do you exclude, and why each one? How many usable days remain?*

In [ ]:
def build_features(frame, p=P, scale=TARGET_SCALE):
    """
    Build strictly causal covariates and the normalised target.

    Parameters
    ----------
    frame : pandas.DataFrame
        Raw daily data, sorted by date.
    p : int, default P
        Number of demand lags.
    scale : float, default TARGET_SCALE
        Declared scale: y_t = cnt_t / scale.

    Returns
    -------
    X : numpy.ndarray, shape (n, 5 + p)
        Rows x_t = (1, workingday, holiday, sin_annual, cos_annual, y_lag1, ..., y_lag{p}).
    y : numpy.ndarray, shape (n,)
        Normalised target, aligned with X.
    dates : pandas.Series, length n
        Dates of the rows kept (index reset).
    feature_names : list of str
        Names of the columns of X, in order.

    Notes
    -----
    Lag k of day t is the demand observed k days BEFORE t. The first p days, whose history
    is incomplete, are dropped. No information from day t or from the future.
    """
    # To complete
    ...


X, y, dates, feature_names = build_features(daily)
# To complete: print the shapes of X and y, the period covered, and the first 5 rows of X with their names.

In [ ]:
# To complete — Check 1: max_t ||x_t||₂ ≤ R and max_t |y_t| ≤ Y.

# To complete — Check 2 (causality): copy `daily`, add 5,000 to `cnt` on day t0 = 300, rebuild
# the covariates, then check with `assert` statements that:
#   - the rows of X up to and including day t0 are unchanged (mind the offset of P rows);
#   - the next day's row changes (through y_lag1).

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.3 — Train then test, predicting before learning</b><br>
Split chronologically: <b>train = 2011</b>, <b>test = 2012</b>. Write an online loop that, each day, <b>predicts first</b> and then updates with $y_t$. How should hyperparameters be chosen without looking at the test set? Should the model keep learning during the test period? Finally, plot the series again, shading the training and test windows.
</div>

**Your answer:** ...

*State the rules that make the evaluation honest: order of predicting and learning, split, choice of hyperparameters, behaviour of the model during the test period.*

In [ ]:
# To complete: boolean mask `is_train` (dates < 2012-01-01), then
# X_train, y_train, dates_train, X_test, y_test, dates_test (with the index of dates_test reset).
# Print the number of days and the period of each window.


def run_online(model, X, y):
    """
    Online loop "predict, then learn".

    Parameters
    ----------
    model : object
        Provides params() (current parameter), predict(x) and update(x, y).
    X : numpy.ndarray, shape (n, d)
    y : numpy.ndarray, shape (n,)

    Returns
    -------
    preds : numpy.ndarray, shape (n,)
        preds[t] is computed BEFORE y[t] is revealed.
    params : numpy.ndarray, shape (n, dim)
        Copy of the parameter used to predict on each day (a copy, not an alias).
    """
    # To complete
    ...


def squared_loss(pred, y):
    """Squared loss ½ (pred − y)², element-wise (works on arrays)."""
    # To complete
    ...


def newsvendor_loss(q, y, c_b=C_B, c_h=C_H):
    """Newsvendor loss c_h (q − y)_+ + c_b (y − q)_+, element-wise (works on arrays)."""
    # To complete
    ...


def select_on_train(make_model, grid, loss):
    """
    Choose a hyperparameter using the training window only.

    Parameters
    ----------
    make_model : callable
        make_model(h) returns a fresh model for hyperparameter h.
    grid : list
        Candidate values.
    loss : callable
        loss(preds, y) returns element-wise losses.

    Returns
    -------
    best : element of grid
        Value with the smallest online average loss on the training window (the first one if tied).
    scores : pandas.Series
        Online average loss on the training window, indexed by the values of grid.
    """
    # To complete
    ...


def train_then_test(make_model, h):
    """
    Learn online on the training window, then continue on the test window without resetting the model.

    Returns
    -------
    preds, params : outputs of run_online on the test window.
    """
    # To complete
    ...


results = {}  # model name → {"pred": ..., "params": ..., "hyper": ...} on the test window (names: see COLORS)

In [ ]:
# To complete: plot the series again (in bikes / day) with a filled area under the curve (fill_between),
# coloured PALETTE["teal"] on the training window and PALETTE["orange"] on the test window,
# a vertical line on 1 January 2012, the y-axis starting at 0, a title and a legend
# giving the number of days in each window.

**Your answer:** ...

*What does this figure show about the difference between the two windows? Why is it an argument for continuing to learn during the test period?*

## 2.4 Model (i-a) — OGD on the squared loss

We start from $\theta_1 = 0 \in \mathcal K$ with a constant step $\eta$, as in the theorem. The grid contains the theoretical step
$\eta^\star = D/(G\sqrt{T_{\text{train}}})$ and larger steps. The theorem reasons in the worst case; in practice, a larger step
often tracks an evolving demand better.

In [ ]:
def project_ball(theta, radius=B):
    """Euclidean projection of theta onto K = {θ : ||θ||₂ ≤ radius}."""
    # To complete
    ...


class OGD:
    """Projected OGD, squared loss ½(xᵀθ − y)², constant step η, starting from θ₁ = 0."""

    def __init__(self, eta, d=X.shape[1], radius=B):
        # To complete: store eta and radius, initialise self.theta
        ...

    def params(self):
        """Return the current parameter θ_t."""
        # To complete
        ...

    def predict(self, x):
        """Return the prediction x_tᵀθ_t."""
        # To complete
        ...

    def update(self, x, y):
        """θ_{t+1} = Π_K(θ_t − η e_t x_t), with e_t = x_tᵀθ_t − y_t."""
        # To complete
        ...


# To complete — causality test: modify y_test[100], rerun run_online(OGD(0.03), ...) and check with
# `assert` statements that the predictions up to and including day 100 are identical and that later ones change.

eta_star = ...  # to complete: theoretical step D / (G_SQ √T_train)
OGD_GRID = sorted({round(eta_star, 4), 0.003, 0.01, 0.03, 0.1, 0.3})
# To complete: best_eta, scores = select_on_train(...); display scores;
# then pred, params = train_then_test(OGD, best_eta) and store them in results["OGD"]
# (with "hyper": f"η = {best_eta}"). Print the final coefficients with their names.

**Your answer:** ...

*Why can the selected step be larger than the theoretical step η*? Which covariates weigh the most? What practical role does the projection play?*

## 2.5 Model (i-b) — OGD + Ridge

We apply OGD to $\widetilde\ell_t(\theta) = \tfrac12(\mathbf x_t^\top\theta - y_t)^2 + \tfrac\lambda2\|\theta\|_2^2$. The update **shrinks** $\theta$ by a factor
$(1 - \eta_t\lambda)$, then corrects along the direction $\mathbf x_t$. The clock $t$ is **not reset** at the start of the test window.

**The step size.** The course uses $\eta_t = 1/(\lambda t)$. For small $\lambda$, the first steps are huge ($\eta_1 = 1/\lambda$): as long as
$\eta_t\|\mathbf x_t\|^2 > 2$, a correction overshoots its target and the algorithm bounces on the boundary of $\mathcal K$. Since $\|\mathbf x_t\|^2 \le R^2$,
this lasts until $t \approx R^2/(2\lambda)$ days: more than 1,500 days for $\lambda = 0.003$, i.e. the whole series. We therefore **cap** the step:
$$
\eta_t = \min\Big\{\eta_{\max},\ \frac{1}{\lambda t}\Big\}, \qquad \eta_{\max} = \frac{1}{R^2},
$$
a cap set **a priori** from the bound $R$ of Question 2.1. As soon as $t \ge R^2/\lambda$, we recover exactly the course's step;
the cap only adds a constant term to the $\log T$ bound. For small $\lambda$, the capped phase covers the whole series:
OGD + Ridge then behaves like OGD with step $1/R^2$ and a slight shrinkage.

We choose $\lambda$ on the training window with the **unpenalised** squared error, since forecast quality is what we care about.
The regret of Part 3, however, is computed on the **penalised** loss, the one in the theorem.

In [ ]:
class OGDRidge(OGD):
    """
    Projected OGD on the ridge loss ½(xᵀθ − y)² + λ/2 ||θ||², with the capped step
    η_t = min(eta_max, 1/(λ t)) and eta_max = 1/R². The clock t is not reset at the test window.
    """

    def __init__(self, lam, d=X.shape[1], radius=B, eta_max=1 / R**2):
        # To complete: call the parent constructor, store lam and eta_max, initialise the clock t
        ...

    def update(self, x, y):
        """θ_{t+1} = Π_K((1 − η_t λ) θ_t − η_t e_t x_t)."""
        # To complete
        ...


RIDGE_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
# To complete: select λ on the training window (UNPENALISED squared loss), then train_then_test;
# store in results["OGD + Ridge"] with the extra key "lam". Compare the final ||θ|| with OGD's.

**Your answer:** ...

*What happens for small λ? For large λ? (Try `eta_max=np.inf`.) Does the log T guarantee concern the forecast error?*

## 2.6 Model (ii) — Newsvendor loss

### Empirical check: the optimum is the quantile $\tau = c_b/(c_b + c_h)$

On the training window, we compute the average cost $\frac1n\sum_t \ell_t(q)$ of a **constant** order $q$ over a fine grid,
and compare its minimiser with the empirical $\tau$-quantile of $(y_t)$. We repeat the experiment for several pairs $(c_b, c_h)$.

In [ ]:
q_grid = np.linspace(0.0, Q_MAX, 4001)
# To complete:
# 1) average cost on the training window of a constant order q, for each q in q_grid;
# 2) its numerical argmin, and the empirical τ-quantile of y_train (np.quantile, method="inverted_cdf");
# 3) plot the cost (in bikes) against q (in bikes), with three vertical lines:
#    numerical argmin, empirical quantile, mean of y_train;
# 4) repeat the argmin / quantile comparison for (c_b, c_h) ∈ {(1, 1), (1, 3), (3, 1), (9, 1)} and display it as a table.

**Your answer:** ...

*Does the argmin coincide with the quantile? Why can there be a whole interval of minimisers? What do we recover for c_b = c_h? Is ordering the mean optimal?*

### Newsvendor OGD without covariates

This is the algorithm of slides 63–64: $q_{t+1} = \Pi_{[0, Q_{\max}]}(q_t - \eta\, g_t)$. No covariate is used, but the order
$q_t$ **moves every day**: after a shortage it goes up by $\eta c_b$, after an excess it goes down by $\eta c_h$. The **comparator**, however, is a single
constant order chosen in hindsight: the clipped empirical quantile.

In [ ]:
class NewsvendorOGD:
    """Order q_t ∈ [0, Q_max] without covariates, learned by projected subgradient steps (starting from q₀)."""

    def __init__(self, eta, q0=0.0, q_max=Q_MAX):
        # To complete
        ...

    def params(self):
        """Return np.array([q_t])."""
        # To complete
        ...

    def predict(self, x):
        """Return q_t (ignores the covariates)."""
        # To complete
        ...

    def update(self, x, y):
        """g_t = c_h if excess, −c_b if shortage, 0 if equal; q_{t+1} = Π_[0, Q_max](q_t − η g_t)."""
        # To complete
        ...


eta_nv_star = ...  # to complete: Q_MAX / (G_NV √T_train)
NV_GRID = sorted({round(eta_nv_star, 4), 0.001, 0.003, 0.01, 0.03})
# To complete: selection on the training window with newsvendor_loss, train_then_test, results["Newsvendor (no covariates)"],
# and the share of 2012 days without shortage P(y_t ≤ q_t), to compare with τ.

### Contextual newsvendor OGD: a conditional quantile

We replace the order without covariates by $q_t = \mathbf x_t^\top\theta_t$, with $\theta_t \in \mathcal K$. The subgradient becomes
$g_t = \big(c_h\,\mathbf 1\{q_t > y_t\} - c_b\,\mathbf 1\{q_t < y_t\}\big)\,\mathbf x_t$. We thus learn a **conditional quantile**
of level $\tau$ that follows the calendar and the recent demand.

*Remark.* Nothing forces $q_t \in [0, Q_{\max}]$ for a linear model. We check it afterwards; in deployment, the order would be clipped.

In [ ]:
class ContextualNewsvendorOGD(OGD):
    """q_t = x_tᵀθ_t; subgradient (c_h 1{q_t > y_t} − c_b 1{q_t < y_t}) x_t; projection onto K."""

    def update(self, x, y):
        # To complete
        ...


CTX_GRID = sorted({round(D / (G_CTX * np.sqrt(len(y_train))), 4), 0.0003, 0.001, 0.003, 0.01})
# To complete: selection, train_then_test, results["Newsvendor (contextual)"], P(y_t ≤ q_t) on 2012,
# and the share of orders outside [0, Q_max].

**Your answer:** ...

*Do both newsvendor policies cover about τ of the days? What does the context add?*

## 2.7 Naive baselines

An algorithm is only worth something compared with a simple baseline. We add two predictors that learn nothing:
**yesterday** ($\widehat y_t = y_{t-1}$) and **last week** ($\widehat y_t = y_{t-7}$). Yesterday corresponds to the fixed parameter
$\theta = e_{\texttt{y\_lag1}}$, with norm $1 \le B$: it is a point of $\mathcal K$, hence a legitimate competitor of the comparator.

In [ ]:
class LagBaseline:
    """Predict the value observed k days earlier (column y_lag{k} of x_t), without learning."""

    def __init__(self, k):
        # To complete: find the index of the column f"y_lag{k}" in feature_names
        ...

    def params(self):
        """No parameter: return np.zeros(0)."""
        # To complete
        ...

    def predict(self, x):
        # To complete
        ...

    def update(self, x, y):
        pass


# To complete: run both baselines on the test window and store them in results["Naive (yesterday)"] and results["Naive (D-7)"]
# (with "hyper": "—").

ORDER = ["Naive (yesterday)", "Naive (D-7)", "OGD", "OGD + Ridge", "Newsvendor (no covariates)", "Newsvendor (contextual)"]
results = {name: results[name] for name in ORDER}

# To complete: plot on 2012 the observed series (light grey) and the predictions of OGD, OGD + Ridge
# and the contextual newsvendor (colours COLORS, styles STYLES).

# Part 3 — Final metrics

All metrics are computed on the **366 days of 2012**, using predictions made before each outcome was revealed.
Errors are converted back into **bikes per day**. Bias is *predicted − observed*: positive means overestimation.
We add the newsvendor cost and the coverage $\mathbb P(y_t \le \widehat y_t)$, which should be about $\tau = 0.75$ for a good quantile.

In [ ]:
def final_metrics(pred, y):
    """
    Test metrics, with errors converted back into bikes (multiply by TARGET_SCALE).

    Returns
    -------
    metrics : dict
        "RMSE", "MAE", "Bias (predicted − observed)", "Newsvendor cost / day" and "Coverage P(y ≤ ŷ)".
    """
    # To complete
    ...


# To complete: `scorecard`, a DataFrame with one row per model (order of results), the metrics above
# and a first column "hyperparameter". Display it.

In [ ]:
# To complete: two horizontal bar charts side by side, one bar per model in its COLORS colour:
#   RMSE on 2012 on the left, newsvendor cost / day on the right. Print the value at the end of each bar.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 3.1 — Is RMSE enough to rank the models?</b><br>
The newsvendor models do not have the lowest RMSE. Is this a failure? What need does each family of models answer, and with which metric should each be judged? Also compare OGD with the naive baselines.
</div>

**Your answer:** ...

## Average regret over time

For each prefix of $n$ test days, we compute
$$
\frac{R_n}{n} = \frac1n\Big(\sum_{j=1}^n \ell_j(\theta_j) - \min_{u \in \mathcal K}\sum_{j=1}^n \ell_j(u)\Big),
$$
**each model with its own loss**, i.e. the loss of its theorem. The comparator is **a single** fixed parameter per prefix, recomputed for each $n$.

- **Squared (and ridge):** $\min_{\|u\|\le B} \tfrac12 u^\top A u - b^\top u$ with $A = \sum \mathbf x_j\mathbf x_j^\top$ (plus $n\lambda I$ for ridge)
  and $b = \sum y_j \mathbf x_j$. If the least-squares solution lies in the ball, it is the answer. Otherwise, the KKT conditions give
  $(A + \mu I)u = b$ with $\|u\| = B$ and $\mu > 0$, and $\mu$ is found by root finding.
- **Newsvendor without covariates:** the comparator is a constant order; the cost is piecewise linear, so the optimum is reached at $0$, at $Q_{\max}$ or at an observed value.
- **Contextual newsvendor:** quantile regression, solved as a linear programme. We check that the solution lies in the ball.
  If it does not, the computed comparator is too good and the curve overestimates the regret: it is then a conservative upper bound.

In [ ]:
def constrained_quadratic_min(A, b, radius=B):
    """
    argmin_{||u||₂ ≤ radius} ½ uᵀAu − bᵀu, for a symmetric positive semi-definite A.

    Hints: diagonalise A = V diag(w) Vᵀ. If the minimum-norm solution of Au = b exists and lies in the ball,
    it is the answer. Otherwise, find μ > 0 such that ||(A + μI)⁻¹ b|| = radius (brentq), then u = (A + μI)⁻¹ b.
    """
    # To complete
    ...


def avg_regret_quadratic(X, y, pred, params=None, lam=0.0):
    """
    Average regret R_n / n at every prefix n, for the squared loss (lam = 0) or the ridge loss (lam > 0).

    The comparator of prefix n minimises over K the sum of the losses of the first n days:
    ½ uᵀA_n u − b_nᵀu + ½ Σ y_j², with A_n = Σ x_j x_jᵀ (+ nλI for ridge) and b_n = Σ y_j x_j.
    For ridge, the algorithm's loss includes λ/2 ||θ_j||² (params[j] = θ_j).

    Returns
    -------
    out : numpy.ndarray, shape (n,)
    """
    # To complete
    ...


def avg_regret_newsvendor_constant(y, q):
    """
    R_n / n against the best constant order in [0, Q_max].
    Hint: the cost is piecewise linear, so the optimum is at 0, at Q_max or at an observed value.
    """
    # To complete
    ...


def avg_regret_newsvendor_contextual(X, y, q):
    """
    R_n / n against the best fixed linear quantile regression, solved as a linear programme:
    min c_b Σ u⁺ + c_h Σ u⁻ subject to x_jᵀθ + u⁺_j − u⁻_j = y_j, u⁺, u⁻ ≥ 0 (scipy.optimize.linprog, method="highs").

    Returns
    -------
    out : numpy.ndarray, shape (n,)
    inside : numpy.ndarray of booleans, shape (n,)
        True if the prefix solution θ* satisfies ||θ*|| ≤ B (otherwise the curve overestimates the regret).
    """
    # To complete
    ...


# To complete: dictionary `avg_regret` (name → curve) for "Naive (yesterday)", "OGD", "OGD + Ridge" (ridge loss),
# "Newsvendor (no covariates)" and "Newsvendor (contextual)". Print the share of prefixes where the contextual
# comparator lies in K.

In [ ]:
# To complete: two panels side by side, average regret R_n / n against the test date, with a horizontal line at 0.
#   left: Naive (yesterday), OGD, OGD + Ridge (write "ridge loss" in the legend);
#   right: both newsvendor models.
# Also display the final average regret of each model.

**Your answer:** ...

*Can regret be negative? Why is that of the newsvendor without covariates so negative? Why does the comparator depend on the model class? Does a low average regret imply a low error? What can a finite curve prove?*

## Conclusion

The cell below summarises **your** numerical results. The take-home messages that follow do not depend on the exact figures.

In [ ]:
best_rmse = scorecard["RMSE"].astype(float).idxmin()
best_cost = scorecard["Newsvendor cost / day"].astype(float).idxmin()
ogd_vs_naive = scorecard.loc["OGD", "RMSE"] / scorecard.loc["Naive (yesterday)", "RMSE"] - 1
print(f"• Best RMSE on 2012: {best_rmse} ({scorecard.loc[best_rmse, 'RMSE']:,.0f} bikes / day).")
print(f"• OGD against yesterday's value: RMSE {ogd_vs_naive:+.1%}.")
print(f"• Lowest newsvendor cost: {best_cost} ({scorecard.loc[best_cost, 'Newsvendor cost / day']:,.0f} / day).")
print(f"• Coverage P(y ≤ q): no covariates = {scorecard.loc['Newsvendor (no covariates)', 'Coverage P(y ≤ ŷ)']:.2f}, "
      f"contextual = {scorecard.loc['Newsvendor (contextual)', 'Coverage P(y ≤ ŷ)']:.2f} (target {TAU}).")
print("• Final average regret: " + ", ".join(f"{k} {v[-1]:+.4f}" for k, v in avg_regret.items()) + ".")

**Your conclusion:** ...

*In at most five points: what the loss changes in the decision; what all the algorithms have in common; how you obtained
the assumptions of the theorems; what makes the protocol honest; the limits of static regret on this series.*